In [1]:
import json
import urllib.request
import urllib.error
from datetime import datetime

In [2]:
PYPI_URL = "https://pypi.org/pypi"
USER_AGENT = "RequirementsValidator/1.0"

In [3]:
def get_json(url):
    """Fetch JSON from PyPI. Return None for HTTP 404."""
    request = urllib.request.Request(
        url,
        headers={"User-Agent": USER_AGENT}
    )

    try:
        with urllib.request.urlopen(request, timeout=10) as response:
            return json.loads(response.read().decode("utf-8"))

    except urllib.error.HTTPError as error:
        if error.code == 404:
            return None
        raise

In [4]:
def validate_requirements_file(filepath="requirements.txt"):
    # Read and parse the requirements file.
    dependencies = []

    try:
        with open(filepath, "r", encoding="utf-8") as file:
            for line in file:
                line = line.strip()

                # Ignore blank lines and comments.
                if not line or line.startswith("#"):
                    continue

                # Handle trailing backslashes.
                line = line.rstrip("\\").strip()

                # Only process pinned dependencies.
                if "==" not in line:
                    continue

                name, version = line.split("==", 1)
                name = name.strip()
                version = version.strip()

                if name and version:
                    dependencies.append((name, version))

    except OSError as error:
        print(f"Error reading {filepath}: {error}")
        return

    packages_not_in_pypi = 0
    versions_not_in_pypi = 0
    valid_releases = []

    # Validate each pinned dependency.
    for name, version in dependencies:
        package_url = f"{PYPI_URL}/{name}/json"
        version_url = f"{PYPI_URL}/{name}/{version}/json"

        try:
            package_data = get_json(package_url)

            if package_data is None:
                packages_not_in_pypi += 1
                continue

            version_data = get_json(version_url)

            if version_data is None:
                versions_not_in_pypi += 1
                continue

            # Use upload_time from the first URL entry.
            urls = version_data.get("urls", [])

            if urls:
                upload_time = urls[0].get("upload_time")

                if upload_time:
                    release_date = datetime.fromisoformat(
                        upload_time.replace("Z", "+00:00")
                    )

                    valid_releases.append(
                        (release_date, name, version)
                    )

        except (
            urllib.error.URLError,
            TimeoutError,
            json.JSONDecodeError,
            ValueError
        ) as error:
            print(f"Could not check {name}=={version}: {error}")

    # Print the results.
    print(f"Dependencies checked: {len(dependencies)}")
    print(f"Packages not in PyPI: {packages_not_in_pypi}")
    print(f"Versions not in PyPI: {versions_not_in_pypi}")

    if valid_releases:
        release_date, name, version = min(valid_releases)
        print(
            f"Oldest pinned release: "
            f"{name}=={version} ({release_date.year})"
        )
    else:
        print("Oldest pinned release: None")



In [6]:
if __name__ == "__main__":
    validate_requirements_file()

Dependencies checked: 8
Packages not in PyPI: 2
Versions not in PyPI: 1
Oldest pinned release: six==1.10.0 (2015)
